# How Toxic Is X (Twitter)? Analysis

In this notebook we take the tweets we collected with `collect_x.py`, clean them, score them with three methods, and answer our three questions:

1. Which topics are the most toxic, and does that match our group's guess?
2. What kind of toxicity is most common in each topic?
3. What does a simple bad-word list miss compared with AI models?

The scoring step is slow, so we save the scores to `data/scored.csv` and reuse them next time.

> This notebook shows tweet text in its outputs. Before committing, clear them with
> `jupyter nbconvert --clear-output --inplace toxicity_analysis.ipynb`

## 0. Setup

In [ ]:
import html
import json
import re
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path("data")
RAW = DATA / "raw" / "combined.csv"
CLEAN = DATA / "clean.csv"
SCORED = DATA / "scored.csv"
SCORE_INFO = DATA / "scored_info.json"
FIG = Path("figures")
FIG.mkdir(exist_ok=True)
BADWORDS = Path("resources/ldnoobw_en.txt")

THRESHOLD = 0.5
SEED = 42

TOPIC_ORDER = ["us_politics", "kpop", "movies", "tech", "pets", "food"]
TOPIC_NAME = {"us_politics": "US Politics", "kpop": "K-pop", "movies": "Movies",
              "tech": "Tech", "pets": "Pets", "food": "Food"}
GUESS = {"us_politics": "High", "kpop": "High", "movies": "Mid",
         "tech": "Mid", "pets": "Low", "food": "Low"}

pd.set_option("display.max_colwidth", 100)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Load the data

We start from `combined.csv`. It has all posts and replies from the 6 topics in one table, and the `type` column tells them apart.

In [ ]:
raw = pd.read_csv(RAW, dtype={"tweet_id": str, "parent_id": str})
raw["text"] = raw["text"].fillna("")
raw["parent_text"] = raw["parent_text"].fillna("")
print(f"{len(raw):,} tweets")
raw[["topic", "hashtag", "type", "text"]].head()

In [ ]:
raw.pivot_table(index="topic", columns="type", values="tweet_id", aggfunc="count").reindex(TOPIC_ORDER)

## 2. Clean

Many posts are ads or shop links, not people talking. Replies also start with @names that are not part of the message. Our plan:

1. Remove **promo** tweets: a link **and** 3 or more hashtags. We check this before cleaning, while the links are still there.
2. Clean the text: decode `&amp;`, remove the @names at the start of replies, remove links.
3. Remove the same text repeated in one topic (copy-paste fan campaigns, bots).
4. Remove texts shorter than 3 words ("lol", "W"). They are too short to judge.

We keep two versions of the text. `text_clean` has no @names or links, and we use it for the word list and to show tweets. `text_model` replaces them with `@user` and `http`, because that's how Twitter-RoBERTa was trained.

In [ ]:
URL = re.compile(r"https?://\S+")
LEAD_MENTIONS = re.compile(r"^(?:\s*@\w+)+")
MENTION = re.compile(r"@\w+")


def clean_text(t):
    t = html.unescape(t)
    t = LEAD_MENTIONS.sub("", t)
    t = URL.sub("", t)
    t = MENTION.sub("", t)
    return " ".join(t.split())


def model_text(t):
    t = html.unescape(t)
    t = LEAD_MENTIONS.sub("", t)
    t = URL.sub("http", t)
    t = MENTION.sub("@user", t)
    return " ".join(t.split())


df = raw.copy()
df["is_promo"] = df["text"].str.contains(URL) & (df["text"].str.count(r"#\w+") >= 3)
df["text_clean"] = df["text"].map(clean_text)
df["text_model"] = df["text"].map(model_text)
df["parent_clean"] = df["parent_text"].map(clean_text)
df[["type", "is_promo", "text", "text_clean"]].head()

Now we apply the four steps and count how many tweets are left after each one.

In [ ]:
steps = {"1. collected": df.groupby(["topic", "type"]).size()}

df = df[~df["is_promo"]]
steps["2. no promo"] = df.groupby(["topic", "type"]).size()

df = (df.assign(_key=df["text_clean"].str.lower())
        .drop_duplicates(["topic", "type", "_key"]).drop(columns="_key"))
steps["3. no duplicates"] = df.groupby(["topic", "type"]).size()

df = df[df["text_clean"].str.split().str.len() >= 3]
steps["4. 3+ words"] = df.groupby(["topic", "type"]).size()

clean = df.reset_index(drop=True)
assert clean["tweet_id"].is_unique
clean[["topic", "level", "hashtag", "type", "tweet_id", "parent_id", "created_at",
       "text_clean", "text_model", "parent_clean"]].to_csv(CLEAN, index=False, encoding="utf-8")

print(f"Kept {len(clean):,} of {len(raw):,} tweets")
(pd.DataFrame(steps).unstack("type").fillna(0).astype(int)
   .reindex(TOPIC_ORDER).rename(index=TOPIC_NAME))

The promo filter removes most **posts** in pets, food and K-pop, which were mostly ads. Replies are almost untouched, because people write them, not shops.

## 3. Score

We score every tweet with three methods:

| Method | Idea | Output |
|---|---|---|
| Bad-word list | Is any word from the LDNOOBW list in the tweet? No learning | number of bad words |
| Detoxify (unbiased) | AI model trained on online comments | 7 scores from 0 to 1 (see below) |
| Twitter-RoBERTa | AI model trained on tweets | 1 score from 0 to 1 (offensive) |

A tweet counts as toxic when the score is 0.5 or more (or, for the list, when it has at least one bad word).

**The 7 Detoxify scores.** Each score is separate, from 0 to 1, so one tweet can be high on several at once (for example, both insult and obscene).

| Score | What it means | How we use it |
|---|---|---|
| `toxicity` | Rude or hateful in general | **Toxic or not** (>= 0.5). Used for Question 1 and 3 |
| `severe_toxicity` | Very hateful or aggressive | Not used. It says *how* toxic, not *what kind* |
| `obscene` | Swearing, vulgar words | **Type of toxicity** (Question 2) |
| `insult` | Insulting a person | **Type of toxicity** (Question 2) |
| `threat` | Wanting to hurt someone | **Type of toxicity** (Question 2) |
| `identity_attack` | Attacking a group (race, religion, gender, ...) | **Type of toxicity** (Question 2) |
| `sexual_explicit` | Sexual content | **Type of toxicity** (Question 2) |

So we use `toxicity` to decide **whether** a tweet is toxic, and then, only for toxic tweets, the highest of the 5 type scores to decide **what kind** it is. (The model also gives 9 more scores for which group is mentioned, like `muslim` or `female`. We don't need those, so we drop them.)

**Twitter-RoBERTa** gives 2 scores, `non-offensive` and `offensive`, which add up to 1. We use only `offensive`.

### 3.1 Bad-word list

We use the English list from **LDNOOBW** (*List of Dirty, Naughty, Obscene, and Otherwise Bad Words*, https://github.com/LDNOOBW/List-of-Dirty-Naughty-Obscene-and-Otherwise-Bad-Words), saved as `resources/ldnoobw_en.txt` (403 words and phrases). It is one of the most widely used bad-word lists on GitHub (about 3.5k stars). We also tested two other lists, [badwordslist](https://code.google.com/archive/p/badwordslist/) (Google Code) and [google-profanity-words](https://github.com/coffee-and-fun/google-profanity-words) (coffee-and-fun). They found a few more toxic tweets, but they also flagged harmless words, such as food ("turkey breasts") and casual words ("hell yeah"). All three lists still missed more than half of the toxic tweets, so our answer to Question 3 does not depend on which list we use.

We match whole words, not parts of words. So "class" doesn't match "ass".

In [ ]:
from nltk.tokenize import TweetTokenizer

words = [w.strip().lower() for w in BADWORDS.read_text(encoding="utf-8").splitlines() if w.strip()]
BW_SINGLE = {w for w in words if " " not in w}
BW_PHRASES = sorted((w for w in words if " " in w), key=len, reverse=True)
BW_PHRASE_RE = re.compile(r"\b(" + "|".join(map(re.escape, BW_PHRASES)) + r")\b")
tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True)


def badword_hits(text):
    t = text.lower().replace("#", "")
    hits = [w for w in tokenizer.tokenize(t) if w in BW_SINGLE]
    hits += BW_PHRASE_RE.findall(t)
    return hits


print(f"{len(BW_SINGLE)} words and {len(BW_PHRASES)} phrases in the list")
for s in ["This game is fucking awesome!", "You are so stupid, uninstall the game", "first class ticket"]:
    print(f"{s!r} -> {badword_hits(s)}")

### 3.2 AI models

We load the Detoxify `unbiased` model weights directly with `transformers` (the `detoxify` package doesn't install cleanly with our versions). Detoxify gives several labels at once, so we use **sigmoid**. RoBERTa picks one of two classes, so we use **softmax**.

If `scored.csv` already has exactly the same tweets as `clean.csv`, we just load it. If the cleaning changed, we score again.

In [ ]:
DETOX_ID = "unitary/unbiased-toxic-roberta"
ROBERTA_ID = "cardiffnlp/twitter-roberta-base-offensive"
DETOX_COLS = ["toxicity", "severe_toxicity", "obscene", "identity_attack",
              "insult", "threat", "sexual_explicit"]


def run_model(model_id, texts, activation, batch_size=32):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id).eval()
    labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
    out, t0 = [], time.time()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tok(texts[i:i + batch_size], padding=True, truncation=True,
                      max_length=128, return_tensors="pt")
            logits = model(**enc).logits
            probs = torch.sigmoid(logits) if activation == "sigmoid" else torch.softmax(logits, -1)
            out.append(probs.numpy())
    print(f"{model_id}: {len(texts):,} tweets in {time.time() - t0:.0f}s")
    return pd.DataFrame(np.vstack(out), columns=labels)


def score_all(clean):
    s = clean.copy()
    s["bw_words"] = s["text_clean"].map(lambda t: "|".join(badword_hits(t)))
    s["bw_count"] = s["bw_words"].map(lambda x: len(x.split("|")) if x else 0)
    texts = s["text_model"].tolist()
    detox = run_model(DETOX_ID, texts, "sigmoid")[DETOX_COLS]   # skip the identity-group labels
    s[[f"detox_{c}" for c in DETOX_COLS]] = detox.to_numpy()
    s["roberta_offensive"] = run_model(ROBERTA_ID, texts, "softmax")["offensive"].to_numpy()
    return s


clean = pd.read_csv(CLEAN, dtype={"tweet_id": str, "parent_id": str}).fillna({"parent_clean": "", "parent_id": ""})
scored = None
if SCORED.exists():
    old = pd.read_csv(SCORED, dtype={"tweet_id": str, "parent_id": str})
    if len(old) == len(clean) and set(old["tweet_id"]) == set(clean["tweet_id"]):
        scored = old
        print(f"Loaded saved scores ({len(scored):,} tweets). Delete {SCORED} to score again.")

if scored is None:
    scored = score_all(clean)
    scored.to_csv(SCORED, index=False, encoding="utf-8")
    import torch, transformers
    SCORE_INFO.write_text(json.dumps({
        "date": datetime.now().isoformat(timespec="seconds"),
        "detoxify_model": DETOX_ID, "roberta_model": ROBERTA_ID,
        "transformers": transformers.__version__, "torch": torch.__version__,
        "n_tweets": len(scored)}, indent=2))

scored = scored.fillna({"bw_words": "", "parent_id": "", "parent_clean": ""})
scored["detox_toxic"] = (scored["detox_toxicity"] >= THRESHOLD).astype(int)
scored["roberta_toxic"] = (scored["roberta_offensive"] >= THRESHOLD).astype(int)
scored["bw_toxic"] = (scored["bw_count"] > 0).astype(int)
scored[["topic", "type", "bw_count", "detox_toxicity", "detox_insult", "roberta_offensive"]].head()

Before we trust the scores, we do a quick sanity check: is the scoring working the right way round? We take two tweets from our own data whose answer is obvious to any person: one clearly toxic (a personal attack) and one clearly friendly (a kind message to someone whose pet died). The toxic one should score high and the friendly one low. If not, there is a bug, for example reading the wrong output column.

In [ ]:
for name, needle in [("toxic example", "YOU ARE THE DUMBEST"), ("friendly example", "RIP SWEET ANGEL")]:
    r = scored[scored["text_clean"].str.contains(needle, case=False, regex=False)].iloc[0]
    print(f"{name:16}: Detoxify {r['detox_toxicity']:.2f}, RoBERTa {r['roberta_offensive']:.2f}, bad words {r['bw_count']}")

### 3.3 First look: results of each method

Before the detailed questions, here are the three methods side by side: the % of toxic tweets per topic, with the number of toxic tweets and all tweets in brackets. Small groups (like Pets replies, 53 tweets) can change a lot with one tweet.

Each method has its own rule for "toxic":

| Method | A tweet is toxic when |
|---|---|
| Detoxify | `toxicity` score >= 0.5 |
| RoBERTa | `offensive` score >= 0.5 |
| Word list | it has at least 1 word from the list |

0.5 is the usual default for these models.

In [ ]:
methods = {"Detoxify": "detox_toxic", "RoBERTa": "roberta_toxic", "Word list": "bw_toxic"}


def method_table(kind):
    g = scored[scored["type"] == kind]
    k = g.groupby("topic")[list(methods.values())].sum().reindex(TOPIC_ORDER)
    n = g.groupby("topic").size().reindex(TOPIC_ORDER)
    return pd.DataFrame({name: [f"{100 * k.loc[t, col] / n[t]:.1f}% ({k.loc[t, col]}/{n[t]})" for t in TOPIC_ORDER]
                         for name, col in methods.items()},
                        index=pd.Index([TOPIC_NAME[t] for t in TOPIC_ORDER], name="topic"))


print("Toxic tweets in total: " + ", ".join(f"{name} {scored[col].sum()}" for name, col in methods.items())
      + f" (of {len(scored):,})")
print()
print("Replies: % toxic (toxic/all)")
display(method_table("reply"))
print("Posts: % toxic (toxic/all)")
display(method_table("post"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)   # same y-axis, so the two panels compare directly
for ax, (kind, title) in zip(axes, [("reply", "Replies"), ("post", "Posts")]):
    pct = (scored[scored["type"] == kind].groupby("topic")[list(methods.values())].mean()
           .reindex(TOPIC_ORDER) * 100)
    for i, (name, color) in enumerate([("Detoxify", "#c0392b"), ("RoBERTa", "#2980b9"), ("Word list", "#7f8c8d")]):
        ax.bar(np.arange(6) + (i - 1) * 0.27, pct[methods[name]], 0.27, label=name, color=color)
    n = scored[scored["type"] == kind].groupby("topic").size()
    ax.set_xticks(np.arange(6), [f"{TOPIC_NAME[t]}\nn={n[t]}" for t in TOPIC_ORDER])
    ax.set_title(f"{title}: % toxic by method")
axes[0].set_ylabel("% toxic tweets")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG / "fig1_toxic_by_topic.png", bbox_inches="tight")
plt.show()

**What we see**

- **All three methods put US Politics first**, for replies and for posts. For replies: Detoxify 21.4%, RoBERTa 32.4%, word list 11.0%.
- **Detoxify and RoBERTa give the same order for replies:** US Politics > K-pop > Pets > Movies > Food > Tech. RoBERTa flags more tweets overall (160 vs 122).
- **The word list finds almost nothing outside politics:** 0–2 replies per topic (0–1.9%). So its order for the other 5 topics depends on 1 or 2 tweets.
- **Posts are closer together.** Apart from US Politics (20–24% with the AI models), all topics are between 2% and 5%, so their order can change with one tweet.

## 4. Question 1: Which topics are the most toxic?

**Replies are our main result**, because that's where people talk to each other. We answer in two steps:

1. **Threshold check.** We count toxic tweets per topic with Detoxify and RoBERTa at 3 thresholds: 0.2, 0.5 and 0.8. If the topic order stays about the same, the result doesn't depend on our choice of 0.5. The word list has no score, only a word count, so a threshold doesn't apply to it.
2. **Our guess vs. the data.** Our vote put 2 topics in each level, so we do the same with the data: sort the 6 topics by % toxic, then top 2 = **High**, middle 2 = **Mid**, bottom 2 = **Low**. The levels are relative because there is no standard cut-off (like "10% = High"), and the methods give very different %. Then we count how many topics match our guess.

In [ ]:
def to_levels(pct):
    """Top 2 topics = High, middle 2 = Mid, bottom 2 = Low (the same way we voted)."""
    order = pct.sort_values(ascending=False, kind="stable").index   # most toxic first
    return pd.Series(["High"] * 2 + ["Mid"] * 2 + ["Low"] * 2, index=order)


def n_match(levels):
    return sum(levels[t] == GUESS[t] for t in TOPIC_ORDER)


SCORES = {"Detoxify": "detox_toxicity", "RoBERTa": "roberta_offensive"}
rows = []
for kind in ["reply", "post"]:
    g = scored[scored["type"] == kind]
    for name, col in SCORES.items():
        for t in [0.2, 0.5, 0.8]:
            pct = ((g[col] >= t).groupby(g["topic"]).mean() * 100).reindex(TOPIC_ORDER)
            rows.append({"type": kind, "method": name, "threshold": t,
                         **{TOPIC_NAME[x]: round(pct[x], 1) for x in TOPIC_ORDER},
                         "match": f"{n_match(to_levels(pct))}/6"})
print("% toxic per topic at each threshold")
pd.DataFrame(rows).set_index(["type", "method", "threshold"])

**Which pair of thresholds is best?** Detoxify and RoBERTa give very different scores, so the same threshold may not suit both. First we look at how the scores are spread, then we try every pair of 0.2, 0.5 and 0.8 (9 pairs) and check how much the two models agree.

In [ ]:
THRESHOLDS = [0.2, 0.5, 0.8]
MODEL_COLOR = {"Detoxify": "#c0392b", "RoBERTa": "#2980b9"}

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, (name, col) in zip(axes, SCORES.items()):
    ax.hist(scored[col], bins=np.linspace(0, 1, 21), color=MODEL_COLOR[name], edgecolor="white")
    for t in THRESHOLDS:
        ax.axvline(t, ls="--", lw=1, color="#7f8c8d")
        ax.text(t, 1.02, f"{t} ({(scored[col] >= t).sum()})", transform=ax.get_xaxis_transform(),
                ha="center", fontsize=8, color="#555555")
    ax.set_yscale("log")
    ax.set_xlabel(f"{name} score")
    ax.set_title(f"{name}: how the scores are spread", pad=16)
axes[0].set_ylabel("tweets (log scale)")
plt.tight_layout()
plt.savefig(FIG / "fig5a_score_spread.png", bbox_inches="tight")
plt.show()
print("Numbers above the dashed lines = tweets with a score at or above that threshold")

Now the 9 pairs. For each pair we count the tweets **both** models call toxic, the ones **only Detoxify** or **only RoBERTa** calls toxic, and **Cohen's kappa** (1 = the two models always agree, 0 = they agree no more than by chance).

In [ ]:
from sklearn.metrics import cohen_kappa_score

rows = []
for td in THRESHOLDS:
    for tr in THRESHOLDS:
        d, r = scored["detox_toxicity"] >= td, scored["roberta_offensive"] >= tr
        rows.append({"Detoxify": td, "RoBERTa": tr, "kappa": round(cohen_kappa_score(d, r), 2),
                     "both": int((d & r).sum()), "only Detoxify": int((d & ~r).sum()),
                     "only RoBERTa": int((~d & r).sum())})
print("Threshold pairs, best agreement first")
pd.DataFrame(rows).sort_values("kappa", ascending=False).reset_index(drop=True)

**What this shows**

- **Detoxify is almost yes-or-no.** Most tweets score near 0 or near 1, so moving the threshold from 0.2 to 0.8 only changes the number of toxic tweets from 165 to 89. **RoBERTa is spread out:** 450 tweets at 0.2 but only 50 at 0.8. So the threshold matters much more for RoBERTa.
- **The two models agree most at Detoxify 0.2 + RoBERTa 0.5 (kappa 0.73)**, then at our normal 0.5 + 0.5 (0.70). The pairs with RoBERTa at 0.2 agree least (0.25–0.44), because RoBERTa then flags hundreds of tweets that Detoxify doesn't.
- **Agreeing is not the same as being right.** Both models can make the same mistake. For example, both flag "Such meanies! 😱💪💕" (Detoxify 0.72, RoBERTa 0.51), which is playful.
- **So we keep 0.5 for both models.** It is the usual default, it agrees almost as well as the best pair (0.70 vs 0.73), and both give a 4/6 match with our guess for replies. To know which threshold is really *correct*, we would need people to label the tweets by hand.

In [ ]:
def level_table(kind):
    """% toxic and data level per topic at the normal rule (models >= 0.5, list >= 1 bad word)."""
    g = scored[scored["type"] == kind]
    out = pd.DataFrame({"our guess": [GUESS[t] for t in TOPIC_ORDER]}, index=TOPIC_ORDER)
    matches = {"our guess": ""}
    for name, col in methods.items():   # Detoxify, RoBERTa, Word list
        pct = g.groupby("topic")[col].mean().reindex(TOPIC_ORDER) * 100
        levels = to_levels(pct)
        out[name] = [f"{pct[t]:.1f}% {levels[t]}" + (" ✓" if levels[t] == GUESS[t] else "") for t in TOPIC_ORDER]
        matches[name] = f"{n_match(levels)}/6"
    out.loc["matches"] = matches
    return out.rename(index=TOPIC_NAME)


print("Replies (✓ = same level as our guess)")
display(level_table("reply"))
print("Posts")
display(level_table("post"))

g = scored.groupby("type")["detox_toxic"].mean() * 100
print(f"All topics together (Detoxify): replies {g['reply']:.1f}% toxic, posts {g['post']:.1f}% toxic")

In [ ]:
# Slope chart: our guess (left) vs. the data (right), replies.
# Dashed lines = the 2/2/2 cut (High / Mid / Low). Red line = the topic changed level.
reply = scored[scored["type"] == "reply"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.3), sharey=True)
for ax, (name, col) in zip(axes, methods.items()):
    pct = reply.groupby("topic")[col].mean().reindex(TOPIC_ORDER) * 100
    levels = to_levels(pct)
    left = {t: i for i, t in enumerate(TOPIC_ORDER)}       # TOPIC_ORDER is already High, High, Mid, Mid, Low, Low
    right = {t: i for i, t in enumerate(levels.index)}     # most toxic first
    for t in TOPIC_ORDER:
        same = levels[t] == GUESS[t]
        ax.plot([0, 1], [left[t], right[t]], "-o", lw=1.5 if same else 2.5,
                color="#7f8c8d" if same else "#c0392b", zorder=3)
        ax.text(-0.06, left[t], TOPIC_NAME[t], ha="right", va="center", fontsize=9)
        ax.text(1.06, right[t], f"{TOPIC_NAME[t]} {pct[t]:.1f}%", ha="left", va="center", fontsize=9,
                color="black" if same else "#c0392b")
    for y in [1.5, 3.5]:
        ax.axhline(y, ls="--", lw=0.8, color="#bdc3c7", zorder=1)
    for y, lvl in [(0.5, "High"), (2.5, "Mid"), (4.5, "Low")]:
        ax.text(0.5, y, lvl, ha="center", va="center", fontsize=8, color="#95a5a6", style="italic",
                bbox={"facecolor": "white", "edgecolor": "none", "pad": 1}, zorder=2)
    ax.set_xlim(-0.75, 1.85)
    ax.set_xticks([0, 1], ["Our guess", "Data"])
    ax.set_title(f"{name}: {n_match(levels)}/6 match")
    ax.tick_params(left=False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_visible(False)
axes[0].set_ylim(5.6, -0.6)
axes[0].set_yticks([])
fig.suptitle("Our guess vs. the data (replies). Red = the topic is in a different level than we guessed", fontsize=11)
plt.tight_layout()
plt.savefig(FIG / "fig2_guess_vs_data.png", bbox_inches="tight")
plt.show()

**What this shows**

- **US Politics is #1 in every case:** both models, all 3 thresholds, replies and posts.
- **Replies are stable for Detoxify:** 4/6 at 0.2, 0.5 and 0.8. At 0.5, both models miss the same 2 topics: Tech is less toxic than we guessed (Low, not Mid), and Pets is more toxic (Mid, not Low).
- **RoBERTa changes with the threshold:** 3/6 at 0.2, 4/6 at 0.5, 2/6 at 0.8. At 0.2 it is too loose: it flags 68% of US Politics replies and about a third of the replies in every other topic except Tech. At 0.8 it is too strict: Movies, Tech, Pets and Food have only 0 or 1 toxic replies each, so their order depends on single tweets.
- **Posts are not stable (1–4/6).** At 0.5, apart from US Politics, all topics are between 2% and 6%, so one tweet can change the order. For example, Detoxify at 0.5 has Movies (5.3%, 7 of 133) just above K-pop (5.2%, 6 of 115).
- **The word list matches only 1/6 for replies.** Outside politics it finds 0–2 bad-word replies per topic, so its ranking isn't reliable.

**Answer to Question 1**

- **US Politics is the most toxic topic.** 21.4% of its replies are toxic (37 of 173, Detoxify), and it is #1 for every method and threshold.
- **K-pop is second** (11.8%, 12 of 102). The other four topics are all below 6%.
- **Tech is the least toxic** (0.6%, 1 of 169).
- **Our guess was mostly right:** 4/6 topics for replies, with both models at 0.5, and with Detoxify at every threshold (RoBERTa drops to 3/6 at 0.2 and 2/6 at 0.8). We guessed too high for Tech and too low for Pets.

**Note on Pets.** We collected only 68 Pets replies (other topics had 139–226), and 53 are left after cleaning. Only 3 of them are toxic, so one tweet changes the % by about 2 points, and section 7.2 shows that 2 of the 3 are jokes. So "Pets is more toxic than we guessed" is not a reliable result.

## 5. Question 2: What kind of toxicity?

For each toxic tweet, we take the type with the highest Detoxify score: obscene, insult, threat, identity attack or sexual. We skip `toxicity` and `severe_toxicity`, because they say *how* toxic, not *what kind*. Posts and replies are combined to get more tweets. Topics with a small n are not reliable.

In [ ]:
TYPE_COLS = ["obscene", "insult", "threat", "identity_attack", "sexual_explicit"]
tox = scored[scored["detox_toxic"] == 1].copy()
tox["tox_type"] = [TYPE_COLS[i] for i in tox[[f"detox_{c}" for c in TYPE_COLS]].to_numpy().argmax(1)]
tox[["topic", "tox_type", "text_clean"]].head(3)

In [ ]:
types = pd.crosstab(tox["topic"], tox["tox_type"]).reindex(index=TOPIC_ORDER, columns=TYPE_COLS).fillna(0)
n_toxic = types.sum(axis=1).astype(int)
types_pct = types.div(n_toxic.replace(0, np.nan), axis=0) * 100
TYPE_COLOR = dict(zip(TYPE_COLS, ["#e67e22", "#c0392b", "#8e44ad", "#2c3e50", "#16a085"]))
SMALL = 10   # fewer toxic tweets than this = not reliable, drawn lighter

fig, ax = plt.subplots(figsize=(9, 3.8))
for y, t in enumerate(TOPIC_ORDER):
    left, alpha = 0, 0.35 if n_toxic[t] < SMALL else 1.0
    for c in TYPE_COLS:
        v = types_pct.loc[t, c]
        ax.barh(y, v, left=left, height=0.7, color=TYPE_COLOR[c], alpha=alpha, edgecolor="white",
                label=c if y == 0 else None)
        if v >= 12:
            ax.text(left + v / 2, y, f"{v:.0f}%", ha="center", va="center", fontsize=8,
                    color="white" if alpha == 1 else "black")
        left += v
ax.set_yticks(range(6), [f"{TOPIC_NAME[t]} (n={n_toxic[t]})" for t in TOPIC_ORDER])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("% of toxic tweets (Detoxify, posts and replies)")
ax.set_title(f"Most common type of toxicity per topic (lighter = fewer than {SMALL} toxic tweets)")
ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIG / "fig3_toxicity_types.png", bbox_inches="tight")
plt.show()
types_pct.round(0).assign(n_toxic=n_toxic).rename(index=TOPIC_NAME)

**Answer to Question 2**

- **Insult is the most common type** in the topics with enough toxic tweets: US Politics 75% (of 75 toxic tweets), K-pop 78% (of 18) and Movies 62% (of 13). People mostly attack each other, not swear in general.
- **Obscene (swearing) is second**, at 21–23% in these three topics.
- **Threat, identity attack and sexual content are rare** (8% or less everywhere).
- **Tech, Pets and Food have only 4–7 toxic tweets each**, so their mix is not reliable. Their top type is often obscene (e.g. 3 of 5 in Pets), and section 7.2 shows that many of these are friendly swearing, not attacks.

## 6. Question 3: What does the bad-word list miss?

We compare the bad-word list with **both** AI models (each at 0.5):

- **Miss:** the model says toxic, but there is no word from the list.
- **False alarm:** a word from the list is there, but the model says not toxic.

In [ ]:
bw = scored["bw_toxic"] == 1
rows = []
for name in SCORES:   # Detoxify, RoBERTa
    m = scored[methods[name]] == 1
    rows.append({"model": name, "model says toxic": m.sum(), "list says toxic": bw.sum(),
                 "both say toxic": (m & bw).sum(), "missed by list": (m & ~bw).sum(),
                 "false alarms of list": (bw & ~m).sum()})
compare = pd.DataFrame(rows).set_index("model")
compare["% missed"] = (100 * compare["missed by list"] / compare["model says toxic"]).round(0).astype(int)
display(compare)

fig, ax = plt.subplots(figsize=(9, 2.6))
parts = [("both say toxic", "#7f8c8d", "both say toxic"),
         ("missed by list", "#c0392b", "only the model (list misses it)"),
         ("false alarms of list", "#e6b0aa", "only the list (false alarm)")]
for y, name in enumerate(compare.index):
    left = 0
    for col, color, label in parts:
        v = compare.loc[name, col]
        ax.barh(y, v, left=left, height=0.6, color=color, edgecolor="white", label=label if y == 0 else None)
        ax.text(left + v / 2, y, v, ha="center", va="center", fontsize=9,
                color="white" if color != "#e6b0aa" else "black")
        left += v
ax.set_yticks(range(len(compare)), compare.index)
ax.invert_yaxis()
ax.set_xlabel("toxic tweets")
ax.set_title("Bad-word list vs. AI models")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.32))
plt.tight_layout()
plt.savefig(FIG / "fig4_list_vs_models.png", bbox_inches="tight")
plt.show()

both_models = (scored["detox_toxic"] == 1) & (scored["roberta_toxic"] == 1)
print(f"Both models say toxic but no word from the list: {(both_models & ~bw).sum()} tweets")

Some examples. **Misses** = both models say toxic, but the list finds nothing. **False alarms** = the list finds a word, but neither model says toxic.

In [ ]:
cols = ["topic", "bw_words", "detox_toxicity", "roberta_offensive", "text_clean"]
no_model = (scored["detox_toxic"] == 0) & (scored["roberta_toxic"] == 0)
print("Misses (random 8)")
display(scored[both_models & ~bw].sample(8, random_state=SEED)[cols].round(2))
print("False alarms (all)")
scored[bw & no_model][cols].round(2)

**What we see**

- **The list misses most toxic tweets:** 83 of 122 for Detoxify (68%) and 117 of 160 for RoBERTa (73%). 63 tweets are toxic for **both** models but have no word from the list. Most are insults without swear words (*stupid, pathetic, dumbest, trash*) or forms of swear words that are not on the list (*fucked*, *sh!t*, *fcking*).
- **The list's false alarms are rare:** 8 for Detoxify, 4 for RoBERTa, and only 4 tweets that neither model calls toxic: two 🖕 emoji used as a joke, "made up bollocks", and a tweet that reports a rape accusation without being abusive.
- **So when the list finds a word, it is usually right, but it finds only about a third (Detoxify) or a quarter (RoBERTa) of the toxic tweets.** It knows swear words, not insults.

## 7. A closer look at the tweets

The AI models give **one score for the whole tweet**, so they don't say which word is the problem. To find it, for every toxic tweet (score 0.5 or more) we **remove one word at a time and score the tweet again**. The 2 words whose removal lowers the score most are the tweet's **trigger words**. We skip small words ("the", "to"), @names, links and emoji.

This takes about 8 minutes, so we save the result to `data/trigger_words.csv` and reuse it.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

TRIGGERS = DATA / "trigger_words.csv"
MODELS = {"Detoxify": (DETOX_ID, "detox_toxic"), "RoBERTa": (ROBERTA_ID, "roberta_toxic")}
SKIP = ENGLISH_STOP_WORDS | {"user", "http", "amp", "rt"}


def norm_word(w):
    """'Shit.' -> 'shit'. Returns '' for words we skip."""
    w = w.lower().strip(".,!?;:\"()[]…“”‘’-").lstrip("#")
    if not re.search("[a-z]", w) or "'" in w or "’" in w or w in SKIP:
        return ""
    return w


def find_triggers(name, model_id, tweets, top_n=2):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id).eval()

    def raw_score(texts):   # model output before the 0-1 squashing
        with torch.no_grad():
            lg = model(**tok(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")).logits
        # Detoxify: column 0 is "toxicity". RoBERTa: "offensive" minus "non-offensive".
        return (lg[:, 0] if name == "Detoxify" else lg[:, 1] - lg[:, 0]).numpy()

    rows, t0 = [], time.time()
    for tid, text in zip(tweets["tweet_id"], tweets["text_model"]):
        words = text.split()
        scores = raw_score([text] + [" ".join(words[:i] + words[i + 1:]) for i in range(len(words))])
        drop = {}
        for w, d in zip(words, scores[0] - scores[1:]):
            w = norm_word(w)
            if w and d > 0:
                drop[w] = max(d, drop.get(w, 0))
        for w, d in sorted(drop.items(), key=lambda x: -x[1])[:top_n]:
            rows.append({"model": name, "tweet_id": tid, "word": w, "drop": round(float(d), 3)})
    print(f"{name}: {len(tweets)} toxic tweets in {time.time() - t0:.0f}s")
    return rows


trig = None
if TRIGGERS.exists() and TRIGGERS.stat().st_mtime > SCORED.stat().st_mtime:   # saved after the last scoring
    trig = pd.read_csv(TRIGGERS, dtype={"tweet_id": str})
    print(f"Loaded saved trigger words. Delete {TRIGGERS} to compute again.")
if trig is None:
    rows = []
    for name, (model_id, col) in MODELS.items():
        rows += find_triggers(name, model_id, scored[scored[col] == 1])
    trig = pd.DataFrame(rows)
    trig.to_csv(TRIGGERS, index=False, encoding="utf-8")

In [ ]:
from html import escape
from IPython.display import HTML

TRIG = trig.groupby(["model", "tweet_id"])["word"].agg(set)


def bold(text, words):
    """Escape the tweet for HTML and put the trigger words in bold."""
    text = escape(text)
    if not words:
        return text
    pat = re.compile(r"\b(" + "|".join(map(re.escape, sorted(words, key=len, reverse=True))) + r")\b", re.I)
    return pat.sub(r"<b>\1</b>", text)


def show_tweets(rows, models=("Detoxify", "RoBERTa")):
    """Table of tweets with both scores, bad words, and the trigger words of `models` in bold."""
    words = [set().union(*(TRIG.get((m, i), set()) for m in models)) for i in rows["tweet_id"]]
    t = pd.DataFrame({
        "topic": rows["topic"].map(TOPIC_NAME), "type": rows["type"],
        "Detoxify": rows["detox_toxicity"].map("{:.2f}".format),
        "RoBERTa": rows["roberta_offensive"].map("{:.2f}".format),
        "bad words": rows["bw_words"].fillna("").str.replace("|", ", ", regex=False),
        "text": [bold(x, w) for x, w in zip(rows["text_clean"], words)]})
    display(HTML(t.to_html(index=False, escape=False).replace("<td>", '<td style="text-align:left">')))

### 7.1 The 15 highest-scoring tweets for each model

**Bold** = trigger words of that model. `bad words` = words found by the bad-word list.

In [ ]:
TOP_N = 15
for name, (col, other) in {"Detoxify": ("detox_toxicity", "roberta_offensive"),
                           "RoBERTa": ("roberta_offensive", "detox_toxicity")}.items():
    print(f"Top {TOP_N} by {name}")
    show_tweets(scored.sort_values([col, other], ascending=False).head(TOP_N), models=(name,))

**What we see**

- **Almost all of the top tweets are US Politics:** 14 of Detoxify's top 15 and 13 of RoBERTa's. The two lists share 9 tweets.
- **They are personal attacks.** The bold trigger words show that the models react to insults (*stupid, pathetic, dumbest, idiot, morons*) as much as to swear words. 6 of Detoxify's top 15 have **no word from the bad-word list**, so the list would miss them.
- **Some trigger words are not insults:** *Democrats*, *vote*, *deport*. So the models partly react to political words themselves, which may make US Politics look slightly more toxic.
- **High score does not always mean hostile.** "girl u crazy, crazy ass a bottle of dry water ... cutie" is a joke (a meme using "ass a" for "as a"), but both models give it 0.93–1.00 because "ass" appears 5 times.

### 7.2 The most toxic tweets in each topic

To see **why** each topic got its level in Question 1, we show the 10 tweets with the highest Detoxify score in every topic (posts and replies together). **Bold** = trigger words of either model.

In [ ]:
TOP_TOPIC = 10
reply_levels = to_levels(reply.groupby("topic")["detox_toxic"].mean().reindex(TOPIC_ORDER))
for t in TOPIC_ORDER:
    g = scored[scored["topic"] == t]
    print(f"{TOPIC_NAME[t]}: our guess {GUESS[t]}, data {reply_levels[t]} (replies, Detoxify), "
          f"{g['detox_toxic'].sum()} of {len(g)} tweets toxic")
    show_tweets(g.sort_values(["detox_toxicity", "roberta_offensive"], ascending=False).head(TOP_TOPIC))

**What we see**

- **US Politics and K-pop:** the top 10 are real attacks on people or fans (*stupid cunt*, *#BLACKPINK fans are trash*). One K-pop mistake: the song title "JENNIE — Damn Right" gets Detoxify 0.84 (RoBERTa 0.09).
- **Movies is mixed:** real insults (*colossal incompetent retard*, *scumbags*) next to false alarms: "OMG THEY'RE BABIESSSS" (0.84), a film called *Pathetic Fallacy* (0.76) and a Richard Pryor quote about swearing.
- **Tech has very few toxic tweets.** Only 3 posts score above 0.9. Below that the Detoxify scores drop fast, which is why Tech came out Low. But RoBERTa finds sexist replies that Detoxify misses, e.g. "Girls women empowerment freedom to become hoe and criminal 🤡" (Detoxify 0.07, RoBERTa 0.78). So Detoxify may count Tech as less toxic than it is.
- **Pets (guessed Low, came out Mid):** its Mid level comes from just 3 toxic replies. Only 1 is a real attack: "Get over it cunt.", written to someone whose dog had died. The other 2, "Such meanies!" and "Slap the dog", are jokes under a playful cat post ("It's so unfair....👇😉❤️"). The 2 toxic Pets posts are also pet talk: a kitten who is "a little shit sometimes" and a "tiny pussy cat".
- **Food (Low):** the same pattern. Only "Couldn't pay me to eat that processed shit. Recipe for morons by morons." is an attack. The others are friendly swearing ("Damn these look so good!", 0.98), double meanings ("I'm a slut for dominos pizza") or violent words in a harmless context ("kill two characters", "chop this one").

**Why calm topics still get toxic tweets:** in Pets and Food, the models are fooled by bad words used in a friendly way, words with a double meaning, and violent words in a harmless context. They score the words, not what the person means. The bad-word list has the same problem with *shit*, *pussy* and *slut*. So for calm topics, a few % "toxic" mostly means false alarms, not people attacking each other.